In [ ]:
!pip install stable-baselines3 shimmy>=2.0 --quiet

In [ ]:
#eVTOLs Autonomos Resgatando Pessoas com Aprendizado por Reforço

In [ ]:
import numpy as np
import random
import imageio

# =========================
# CONFIG
# =========================
GRID = 60
SCALE = 10

TREE_SIZE = 1.2

N_DRONES = 3
N_VICTIMS = 5
N_TREES = 6

MAX_STEPS = 2000

BASE = np.array([GRID // 2, GRID // 2], dtype=np.float32)

STEP = 0.7
SENSOR_RANGE = 1.5

REPULSION_RADIUS = 2.5
REPULSION_STRENGTH = 2.5

WAYPOINT_TOL = 2.0

R_MIN = 1.0
R_MAX = GRID // 2 - 2
R_STEP = 0.4

# RL
ACTIONS = [-0.5, -0.2, 0, 0.2, 0.5]
ALPHA = 0.1
GAMMA = 0.9
EPSILON = 0.1

# =========================
# ENV
# =========================
class RescueMission:

    def __init__(self, use_rl=False):
        self.use_rl = use_rl
        self.Q = {}
        self.reset()

    def reset(self):
        self.pos = {i: BASE.copy() for i in range(N_DRONES)}
        self.theta = {i: i * (2*np.pi / N_DRONES) for i in range(N_DRONES)}
        self.radius = {i: 1.0 for i in range(N_DRONES)}
        self.waypoint = {i: BASE.copy() for i in range(N_DRONES)}
        self.radial_dir = {i: 1 for i in range(N_DRONES)}

        self.state = {i: "search" for i in range(N_DRONES)}
        self.target = {i: None for i in range(N_DRONES)}

        self.trees = []
        while len(self.trees) < N_TREES:
            x = random.randint(2, GRID-3)
            y = random.randint(2, GRID-3)
            if (x, y) != tuple(BASE):
                self.trees.append(np.array([x, y], dtype=np.float32))

        self.victims = []
        for _ in range(N_VICTIMS):
            while True:
                v = np.array([
                    random.randint(2, GRID-3),
                    random.randint(2, GRID-3)
                ], dtype=np.float32)

                if np.linalg.norm(v - BASE) > R_MAX:
                    continue

                if any(np.linalg.norm(v - t) < 1.5 for t in self.trees):
                    continue

                self.victims.append(v)
                break

        self.available = set(range(N_VICTIMS))
        self.carrying = {i: None for i in range(N_DRONES)}
        self.rescued = set()
        self.return_all = False

    # =========================
    def get_state(self, i):
        return tuple(np.round(self.pos[i] / 5).astype(int))

    def choose_action(self, state):
        if random.random() < EPSILON:
            return random.choice(ACTIONS)
        return max(ACTIONS, key=lambda a: self.Q.get((state,a),0))

    def update_q(self, s, a, r, s2):
        best = max([self.Q.get((s2,a2),0) for a2 in ACTIONS])
        self.Q[(s,a)] = self.Q.get((s,a),0) + ALPHA*(r + GAMMA*best - self.Q.get((s,a),0))

    # =========================
    def repulsion(self, pos):
        force = np.zeros(2)
        for t in self.trees:
            diff = pos - t
            dist = np.linalg.norm(diff)
            if dist < REPULSION_RADIUS and dist > 1e-5:
                force += (diff/dist) * (REPULSION_STRENGTH / dist)
        return force

    # =========================
    def move(self, i, target):

        desired = target - self.pos[i]
        desired /= (np.linalg.norm(desired) + 1e-8)

        # RL ajuste de ângulo
        if self.use_rl:
            state = self.get_state(i)
            action = self.choose_action(state)

            angle = np.arctan2(desired[1], desired[0])
            angle += action
            desired = np.array([np.cos(angle), np.sin(angle)])

        direction = desired + self.repulsion(self.pos[i])
        direction /= (np.linalg.norm(direction) + 1e-8)

        old_pos = self.pos[i].copy()
        new_pos = self.pos[i] + direction * STEP

        blocked = False
        for t in self.trees:
            if np.linalg.norm(new_pos - t) < 1.0:
                blocked = True
                break

        if not blocked:
            self.pos[i] = new_pos

        # reward RL
        if self.use_rl:
            r = -np.linalg.norm(self.pos[i] - target)
            s2 = self.get_state(i)
            self.update_q(state, action, r, s2)

    # =========================
    def update_spiral(self, i):

        if self.radius[i] >= R_MAX:
            self.radial_dir[i] = -1
        elif self.radius[i] <= R_MIN:
            self.radial_dir[i] = 1

        self.radius[i] += self.radial_dir[i] * R_STEP
        self.theta[i] += 0.5

        self.waypoint[i] = np.array([
            BASE[0] + self.radius[i] * np.cos(self.theta[i]),
            BASE[1] + self.radius[i] * np.sin(self.theta[i])
        ])

    # =========================
    def step(self):

        if len(self.available) == 0 and all(v is None for v in self.carrying.values()):
            self.return_all = True

        for i in range(N_DRONES):

            if self.return_all:
                self.move(i, BASE)
                continue

            if self.state[i] == "search":

                if np.linalg.norm(self.pos[i] - self.waypoint[i]) < WAYPOINT_TOL:
                    self.update_spiral(i)

                self.move(i, self.waypoint[i])

                for j in list(self.available):
                    if np.linalg.norm(self.pos[i] - self.victims[j]) < SENSOR_RANGE:
                        self.target[i] = j
                        self.available.remove(j)
                        self.state[i] = "to_victim"
                        break

            elif self.state[i] == "to_victim":
                v = self.victims[self.target[i]]
                self.move(i, v)

                if np.linalg.norm(self.pos[i] - v) < 0.8:
                    self.carrying[i] = self.target[i]
                    self.state[i] = "to_base"

            elif self.state[i] == "to_base":
                self.move(i, BASE)

                if np.linalg.norm(self.pos[i] - BASE) < 0.8:
                    self.rescued.add(self.carrying[i])
                    self.carrying[i] = None
                    self.state[i] = "search"

        return self.return_all and all(
            np.linalg.norm(self.pos[i] - BASE) < 0.8 for i in range(N_DRONES)
        )

# =========================
# DESENHO (INALTERADO)
# =========================
def draw_tree(img, cx, cy):
    cx = int(cx * SCALE)
    cy = int(cy * SCALE)
    r = int(SCALE * TREE_SIZE)

    for dx in range(-r, r):
        for dy in range(-r, r):
            if dx*dx + dy*dy <= r*r:
                x = cx - r + dx
                y = cy + dy
                if 0 <= x < img.shape[0] and 0 <= y < img.shape[1]:
                    img[x, y] = [20,160,20]

    for dx in range(-r//4, r//4):
        for dy in range(0, r):
            x = cx + dy
            y = cy + dx
            if 0 <= x < img.shape[0] and 0 <= y < img.shape[1]:
                img[x, y] = [120,70,30]


def draw_person(img, x, y):
    x = int(x * SCALE)
    y = int(y * SCALE)

    c = [255,255,255]

    img[x:x+3, y:y+3] = c
    img[x+4:x+8, y+1:y+2] = c
    img[x+4:x+6, y-1:y+1] = c
    img[x+4:x+6, y+2:y+4] = c
    img[x+8:x+11, y:y+1] = c
    img[x+8:x+11, y+2:y+3] = c


def draw_frame(env):
    img = np.zeros((GRID*SCALE, GRID*SCALE, 3), dtype=np.uint8)
    img[:] = 30

    for t in env.trees:
        draw_tree(img, t[0], t[1])

    for i, v in enumerate(env.victims):
        if i in env.available:
            draw_person(img, v[0], v[1])

    bx = int(BASE[0]*SCALE)
    by = int(BASE[1]*SCALE)
    img[bx:bx+SCALE, by:by+SCALE] = [255,0,0]

    for i in range(N_DRONES):
        p = env.pos[i]
        x = int(p[0]*SCALE)
        y = int(p[1]*SCALE)
        color = [0,255,0] if env.carrying[i] else [0,0,255]
        img[x:x+SCALE, y:y+SCALE] = color

    return img

# =========================
# RUN
# =========================
def run(use_rl, filename):
    env = RescueMission(use_rl=use_rl)
    frames = []

    frames.append(draw_frame(env))

    for step in range(MAX_STEPS):
        done = env.step()

        if step % 2 == 0:
            frames.append(draw_frame(env))

        if done:
            break

    imageio.mimsave(filename, frames, fps=8)
    print("Salvo:", filename)


if __name__ == "__main__":
    #run(False, "no_rl.gif")
    run(True, "with_rl.gif")

Salvo: no_rl.gif
Salvo: with_rl.gif


# Nova seção

# Nova seção